# Ridge and Lasso Regression - Car Price Prediction

This notebook trains Ridge and Lasso regression models on the car price dataset. It compares both regularized linear models and saves the best pipeline for the Streamlit app.

In [ ]:
import joblib
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
data = pd.read_csv('CarPrice_Assignment.csv')
data.head()

In [ ]:
data.info()
data.describe()

In [ ]:
data = data.drop(columns=['car_ID', 'CarName'])
X = data.drop(columns=['price'])
y = data['price']

numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

numeric_features, categorical_features

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', StandardScaler(), numeric_features),
        ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical_features),
    ]
)

In [ ]:
models = {
    'Ridge Regression': Pipeline([('preprocessor', preprocessor), ('model', Ridge(random_state=42))]),
    'Lasso Regression': Pipeline([('preprocessor', preprocessor), ('model', Lasso(random_state=42, max_iter=10000))]),
}

param_grid = {'model__alpha': [0.01, 0.1, 1.0, 10.0, 100.0]}
results = {}
best_name = None
best_search = None
best_score = float('-inf')

for name, pipeline in models.items():
    search = GridSearchCV(pipeline, param_grid, cv=5, scoring='r2', n_jobs=-1)
    search.fit(X_train, y_train)
    predictions = search.predict(X_test)
    mse = mean_squared_error(y_test, predictions)
    metrics = {
        'best_alpha': search.best_params_['model__alpha'],
        'cv_r2': search.best_score_,
        'test_r2': r2_score(y_test, predictions),
        'mae': mean_absolute_error(y_test, predictions),
        'rmse': mse ** 0.5,
    }
    results[name] = metrics
    if metrics['test_r2'] > best_score:
        best_score = metrics['test_r2']
        best_name = name
        best_search = search

pd.DataFrame(results).T

In [ ]:
bundle = {
    'model': best_search.best_estimator_,
    'best_model_name': best_name,
    'metrics': results,
    'feature_columns': X.columns.tolist(),
    'input_defaults': X.median(numeric_only=True).to_dict(),
    'category_options': {
        column: sorted(X[column].dropna().unique().tolist())
        for column in X.select_dtypes(include=['object']).columns
    },
}

joblib.dump(bundle, 'ridge_lasso_model.pkl')
best_name